What You're Aiming For

The objective is to automate the extraction of HTML content, article titles, text, and internal links from Wikipedia pages into a consolidated function that accepts any Wikipedia URL for efficient data retrieval and processing.


Instructions

Create a Python script to automate data extraction from Wikipedia pages. The script will retrieve HTML content, extract article titles and text, collect internal links, and consolidate these tasks into one function that accepts a Wikipedia URL. This will be tested on a specific Wikipedia page to validate functionality.

1) Write a function to Get and parse html content from a Wikipedia page

2) Write a function to Extract article title

3) Write a function to Extract article text for each paragraph with their respective

headings. Map those headings to their respective paragraphs in the dictionary.

4) Write a function to collect every link that redirects to another Wikipedia page

5) Wrap all the previous functions into a single function that takes as parameters a Wikipedia link

6) Test the last function on a Wikipedia page of your choice

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin



# Question 1: Write a function to Get and parse html content from a Wikipedia page

def get_html_content(url):
    """
    Retrieve and parse HTML content from a Wikipedia page.
    
    Args:
        url (str): The URL of the Wikipedia page to scrape.
        
    Returns:
        BeautifulSoup: Parsed HTML content as a BeautifulSoup object.
        
    Raises:
        ValueError: If the page cannot be retrieved (non-200 status code).
    """
    response = requests.get(url)

    if response.status_code != 200:
        raise ValueError(f"Error retrieving page: {response.status_code}")
    
    return BeautifulSoup(response.content, 'html.parser')


# Question 2: Write a function to Extract article title

def extract_article_title(soup):
    """
    Extract the title of the Wikipedia article.
    
    Args:
        soup (BeautifulSoup): Parsed HTML content.
        
    Returns:
        str: The title of the article.
    """
    # The title is in the first h1 tag with id="firstHeading"

    title = soup.find('h1', id='firstHeading')
    return title.text if title else "No title found"

# Question 3: Write a function to Extract article text for each paragraph with their respective headings. Map those headings to their respective paragraphs in the dictionary.

def extract_article_text(soup):
    """
    Extract article text organized by headings and paragraphs.
    
    Args:
        soup (BeautifulSoup): Parsed HTML content.
        
    Returns:
        dict: A dictionary mapping headings to lists of paragraphs under each heading.
              The main content (before any headings) is stored under the key "Introduction".
    """
    content = {}
    current_heading = "Introduction"
    content[current_heading] = []
    
    # Find the main content div
    main_content = soup.find('div', id='mw-content-text')
    if not main_content:
        return content
    
    # Get all relevant elements - headings (h2-h6) and paragraphs (p)
    elements = main_content.find_all(['h2', 'h3', 'h4', 'h5', 'h6', 'p'])
    
    for element in elements:
        if element.name.startswith('h'):  # It's a heading
            heading_level = int(element.name[1])
            heading_text = element.get_text().strip()
            
            # Remove [edit] suffix if present
            if heading_text.endswith('[edit]'):
                heading_text = heading_text[:-6].strip()
            
            current_heading = heading_text
            content[current_heading] = []
        elif element.name == 'p':  # It's a paragraph
            paragraph_text = element.get_text().strip()
            if paragraph_text:  # Only add non-empty paragraphs
                content[current_heading].append(paragraph_text)
    
    # Remove headings with no content
    content = {k: v for k, v in content.items() if v}
    
    return content


# Question 4: Write a function to collect every link that redirects to another Wikipedia page

def extract_internal_links(soup, base_url="https://en.wikipedia.org"):
    """
    Extract all internal Wikipedia links from the article.
    
    Args:
        soup (BeautifulSoup): Parsed HTML content.
        base_url (str): The base Wikipedia URL to resolve relative links.
        
    Returns:
        list: A list of absolute URLs that link to other Wikipedia pages.
    """
    internal_links = set()  # Using a set to avoid duplicates
    
    # Find all anchor tags in the main content
    main_content = soup.find('div', id='mw-content-text')
    if not main_content:
        return []
    
    for link in main_content.find_all('a', href=True):
        href = link['href']
        
        # Check if it's an internal Wikipedia link (starts with /wiki/)
        # and not a special Wikipedia page (doesn't contain :)
        if href.startswith('/wiki/') and ':' not in href:
            absolute_url = urljoin(base_url, href)
            internal_links.add(absolute_url)
    
    return sorted(internal_links)

# Question 5: Wrap all the previous functions into a single function that takes as parameters a Wikipedia link

def scrape_wikipedia_article(url):
    """
    Scrape a Wikipedia article and extract its title, text, and internal links.
    
    Args:
        url (str): The URL of the Wikipedia page to scrape.

    Returns:
        dict: A dictionary containing the article title, text organized by headings,
              and a list of internal links.
    """

    try:

        # Get and parse HTML content
        soup = get_html_content(url)
        
        # Extract article title
        title = extract_article_title(soup)
        
        # Extract article text
        text = extract_article_text(soup)
        
        # Extract internal links
        internal_links = extract_internal_links(soup)
        
        return {
            "title": title,
            "text": text,
            "internal_links": internal_links
        }
    except ValueError as e:
        print(f"Error: {e}")
        return None
    
# Example usage
if __name__ == "__main__":
    url = "https://en.wikipedia.org/wiki/Web_scraping"
    result = scrape_wikipedia_article(url)
    
    if result:
        print("Title:", result["title"])
        print("\nText:")
        for heading, paragraphs in result["text"].items():
            print(f"\n{heading}:")
            for paragraph in paragraphs:
                print(paragraph)
        
        print("\nInternal Links:")
        for link in result["internal_links"]:
            print(link)
    else:
        print("Failed to scrape the article.")
        

Title: Web scraping

Text:

Introduction:
Web scraping, web harvesting, or web data extraction is data scraping used for extracting data from websites.[1] Web scraping software may directly access the World Wide Web using the Hypertext Transfer Protocol or a web browser. While web scraping can be done manually by a software user, the term typically refers to automated processes implemented using a bot or web crawler. It is a form of copying in which specific data is gathered and copied from the web, typically into a central local database or spreadsheet, for later retrieval or analysis.
Scraping a web page involves fetching it and then extracting data from it. Fetching is the downloading of a page (which a browser does when a user views a page). Therefore, web crawling is a main component of web scraping, to fetch pages for later processing. Having fetched, extraction can take place. The content of a page may be parsed, searched and reformatted, and its data copied into a spreadsheet o